In [46]:
import torch
from IPython.display import display

from synthetic_data_generation import make_cohort
import pandas as pd

RANDOM_STATE = 0

# 0 Generate Synthetic Data

In [47]:
data_folder = '../../../data/synthetic_data'
make_cohort(data_folder)

In [48]:
tabular_df = pd.read_csv(data_folder + '/clinical.csv', index_col=0)
display(tabular_df.head())

,age,gender,stage
patient_id,,,
patient_000,63.047171,M,late
patient_001,49.600159,F,early
patient_002,NaN,F,early
patient_003,69.405647,M,early
patient_004,40.489648,F,early


# 1 Data Loading

## 1.1 Tabular

## 1.2 Patch Features
Patch Features = Whole Slide Images passed through an encoder such as UNI

## 1.3 The Target

## 1.4 Multimodal Dataset: All Modalities Combined

# 2 Model Building

## 2.1 Intermediate Fusion

In [55]:
import base64
from IPython.display import Image, display

graph = """
flowchart LR
    A[Modality A<br/>e.g. Clinical table] --> EA
    B[Modality B<br/>e.g. Pathology patches] --> EB
    subgraph M[The Model]
        EA[Encoder A]
        EB[Encoder B]
        F((Fusion))
        H[Shared prediction head]
        EA -->|embedding A| F
        EB -->|embedding B| F
        F -->|fused representation| H
    end
    H --> O([Prediction])
"""

encoded = base64.urlsafe_b64encode(graph.encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}"))

In [56]:
# Needed to ensure model weights are initialised consistently
torch.manual_seed(RANDOM_STATE)

# 3 The Model Orchestrator

## 3.1 Creating the Transformation
Trained/fitted transforms must be passed into the `Pipeline` object, and not applied beforehand.

Training/fitting transforms on the full dataset (train + test) is data leakage!

In [61]:
# from the above information we make must make the following transforms.
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

numeric_pipeline = make_pipeline(SimpleImputer(), StandardScaler())

tab_transfromers = ColumnTransformer([
    ('numeric', numeric_pipeline, ['age']),
    ('categorical', OrdinalEncoder(encoded_missing_value=-1), ['gender'])
])

## 3.2 Creating the Orchestrator: `Pipeline`

# 4 Training

## 4.1 Fitting

## 4.2 [OPTIONAL] How are the transformers fitted?

In [65]:
# [OPTIONAL]
# Insight into how the transformers are applied in the Pipeline.fit

fitted_t = tab_transfromers.fit(train.modalities['clinical'].frame)
fitted_train = train.modalities['clinical'].with_transform(fitted_t, train.ids)

print('Number of missing values before transforms:')
print(train.modalities['clinical'].frame.isna().sum().sum())

print()
print('Number of missing values after transforms:')
print(fitted_train.frame.isna().sum().sum())


Number of missing values before transforms:
18

Number of missing values after transforms:
0


# 5 Evaluation
Note that we are using small synthetic illustrative cohort, so results don't provide real clinical insight

## 5.1 Prediction

## 5.2 Evaluation

# 6 [Advanced] Comparing Models

In [71]:
from kalecancer.model import Unimodal
from sklearn.base import clone
torch.manual_seed(0)
clinical_model = Unimodal(
    "clinical",
    encoders=[torch.nn.Linear(dataset.modalities['clinical'].frame.shape[1], 8)],
    head=ClassificationHead(in_dim=8, n_classes=2),
)
baseline_pipeline = clone(pipeline).set_params(model=clinical_model).fit(train)

In [72]:
from kalecancer.evaluate import EvalContext, bootstrap_ci
prediction, baseline_prediction = pipeline.predict(test), baseline_pipeline.predict(test)

target = test.target.frame.loc[prediction.index]
context = EvalContext(train_target=pipeline.train_target_, classes=pipeline.target_.classes)
baseline_context = EvalContext(train_target=baseline_pipeline.train_target_, classes=baseline_pipeline.target_.classes)

intervals = {
    "fused": bootstrap_ci( AUROC(positive_class='late'), prediction, target, context),
    "clinical only": bootstrap_ci( AUROC(positive_class='late'), baseline_prediction, target, baseline_context),
    "gain from the slides": bootstrap_ci(
         AUROC(positive_class='late'), prediction, target, context, baseline=baseline_prediction, baseline_context=baseline_context
    ),
}
table = pd.DataFrame({name: vars(ci) for name, ci in intervals.items()}).T[["estimate", "lower", "upper"]]
# explicit, as not every editor renders a bare DataFrame on the last line (PyCharm, for one)
display(table)

,estimate,lower,upper
fused,0.727273,0.484706,0.939394
clinical only,0.626263,0.353535,0.875000
gain from the slides,0.101010,-0.046966,0.250000
